In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
# Sửa cách import AdamW
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')
import os
from tqdm import tqdm
import random

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

In [2]:
# Configuration
class Config:
    # Model
    model_name = "vinai/phobert-base"  # PhoBERT for Vietnamese
    max_len = 256
    batch_size = 16
    epochs = 5
    learning_rate = 2e-5
    weight_decay = 0.01
    n_folds = 5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_classes = 3  # 0: Negative, 1: Neutral, 2: Positive
    gradient_accumulation_steps = 1
    warmup_ratio = 0.1
    
config = Config()
print(f"Using device: {config.device}")

# Load data
try:
    train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')
except FileNotFoundError:
    # Nếu file không tìm thấy, thử đường dẫn khác
    train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Sentiment distribution:\n{train_df['sentiment'].value_counts().sort_index()}")


Using device: cuda
Train shape: (11322, 4)
Test shape: (4853, 2)
Sentiment distribution:
sentiment
0    5226
1     501
2    5595
Name: count, dtype: int64


In [3]:
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        sentence = str(self.df.iloc[idx]['sentence'])
        
        # Tokenize cho PhoBERT - sử dụng __call__ thay vì encode_plus
        encoding = self.tokenizer(
            sentence,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if self.is_train:
            item['sentiment'] = torch.tensor(self.df.iloc[idx]['sentiment'], dtype=torch.long)
            
        return item

In [4]:
# Model definition
class PhoBERTForSentiment(nn.Module):
    def __init__(self, model_name, num_classes):
        super(PhoBERTForSentiment, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # Use [CLS] token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [5]:
# Training function
def train_epoch(model, data_loader, optimizer, scheduler, criterion, device):
    model.train()
    losses = []
    all_preds = []
    all_labels = []
    
    progress_bar = tqdm(data_loader, desc='Training')
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['sentiment'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        losses.append(loss.item())
        
        _, preds = torch.max(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        progress_bar.set_postfix({'loss': np.mean(losses)})
    
    avg_loss = np.mean(losses)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return avg_loss, accuracy, f1

In [6]:
# Evaluation function
def eval_epoch(model, data_loader, criterion, device):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(data_loader, desc='Evaluating')
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['sentiment'].to(device)
            
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            
            losses.append(loss.item())
            
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = np.mean(losses)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return avg_loss, accuracy, f1

In [7]:
# Prediction function
def predict(model, data_loader, device):
    model.eval()
    all_preds = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(input_ids, attention_mask)
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
    
    return all_preds

In [8]:
# Main training loop with cross-validation
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)

# Prepare test data
test_dataset = SentimentDataset(test_df, tokenizer, config.max_len, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

Loading tokenizer...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# Cross-validation
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)
X = train_df['sentence'].values
y = train_df['sentiment'].values

fold_predictions = []
val_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'='*50}")
    print(f"Fold {fold + 1}/{config.n_folds}")
    print(f"{'='*50}")
    
    # Split data
    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)
    
    # Create datasets
    train_dataset = SentimentDataset(train_fold, tokenizer, config.max_len, is_train=True)
    val_dataset = SentimentDataset(val_fold, tokenizer, config.max_len, is_train=True)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    
    # Initialize model
    print(f"Initializing model for fold {fold + 1}...")
    model = PhoBERTForSentiment(config.model_name, config.num_classes).to(config.device)
    
    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_steps = len(train_loader) * config.epochs
    warmup_steps = int(total_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    criterion = nn.CrossEntropyLoss()
    
    # Training
    best_val_f1 = 0
    best_model_state = None
    
    for epoch in range(config.epochs):
        print(f"\nEpoch {epoch + 1}/{config.epochs}")
        
        train_loss, train_acc, train_f1 = train_epoch(
            model, train_loader, optimizer, scheduler, criterion, config.device
        )
        print(f"Train - Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, F1: {train_f1:.4f}")
        
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, config.device)
        print(f"Val - Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, F1: {val_f1:.4f}")
        
        # Save best model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"New best model! Val F1: {val_f1:.4f}")
    
    # Load best model for this fold
    model.load_state_dict(best_model_state)
    model = model.to(config.device)
    val_scores.append(best_val_f1)
    
    # Predict on test set
    print(f"Predicting on test set for fold {fold + 1}...")
    fold_pred = predict(model, test_loader, config.device)
    fold_predictions.append(fold_pred)

print(f"\nCross-validation scores: {val_scores}")
print(f"Mean CV F1: {np.mean(val_scores):.4f} (+/- {np.std(val_scores):.4f})")



Fold 1/5
Initializing model for fold 1...


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

...

Train - Loss: 0.0825, Accuracy: 0.9792, F1: 0.9791


Evaluating: 100%|██████████| 142/142 [00:30<00:00,  4.59it/s]


Val - Loss: 0.3141, Accuracy: 0.9377, F1: 0.9368
New best model! Val F1: 0.9368
Predicting on test set for fold 5...


Predicting: 100%|██████████| 304/304 [01:06<00:00,  4.60it/s]


Cross-validation scores: [0.9425156078119622, 0.9388042596352844, 0.934400739654883, 0.9371864342723171, 0.9368050310838788]
Mean CV F1: 0.9379 (+/- 0.0027)


In [10]:
# Ensemble predictions (majority voting)
final_predictions = []
fold_predictions = np.array(fold_predictions)

for i in range(fold_predictions.shape[1]):
    pred_counts = np.bincount(fold_predictions[:, i])
    final_predictions.append(np.argmax(pred_counts))

# Create submission file
submission = pd.DataFrame({
    'id': test_df['id'],
    'sentiment': final_predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("\nSubmission saved to submission.csv")
print("\nSentiment distribution in predictions:")
print(submission['sentiment'].value_counts().sort_index())

# Optionally, create an ensemble with weights based on validation scores
if len(val_scores) > 0:
    # Normalize validation scores as weights
    weights = np.array(val_scores) / np.sum(val_scores)
    
    weighted_predictions = []
    for i in range(fold_predictions.shape[1]):
        weighted_votes = np.zeros(config.num_classes)
        for fold_idx in range(len(fold_predictions)):
            weighted_votes[fold_predictions[fold_idx, i]] += weights[fold_idx]
        weighted_predictions.append(np.argmax(weighted_votes))
    
    # Save weighted ensemble submission
    submission_weighted = pd.DataFrame({
        'id': test_df['id'],
        'sentiment': weighted_predictions
    })
    submission_weighted.to_csv('/kaggle/working/submission_weighted.csv', index=False)
    print("\nWeighted ensemble submission saved to submission_weighted.csv")

print("\nDone!")


Submission saved to submission.csv

Sentiment distribution in predictions:
sentiment
0    2258
1     150
2    2445
Name: count, dtype: int64

Weighted ensemble submission saved to submission_weighted.csv

Done!
